# Time Series Practice 3
## Kết quả Phase 1–4

Notebook này chỉ dùng để **trình bày kết quả** từ các artifacts chính thức đã được kiểm chứng. Logic xử lý dữ liệu, tạo feature, tạo `X/y` và temporal split thuộc các module trong `src/practice_3/`; notebook không chạy lại các pipeline đó.

### 1. Bài toán, input và output

- **Bài toán:** chuyển chuỗi mức tiêu thụ điện theo giờ thành supervised learning data phục vụ dự báo bằng Machine Learning.
- **Input:** `datasets/raw/AEP_hourly.csv`, gồm timestamp `Datetime` và target `AEP_MW`.
- **Nguồn kết quả trình bày:** bốn audit JSON của Phase 1–4.
- **Output đến Phase 4:** dữ liệu đã chuẩn bị, sáu feature configurations, sáu bộ `X/y` cùng population và temporal split dùng chung.
- **Chưa thuộc phạm vi:** model training, forecasting và evaluation của Phase 5–7.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
result_dir = project_root / "outputs/results"
processed_path = project_root / "datasets/processed/aep_hourly_prepared.csv"
feature_config_path = project_root / "config/feature_engineering.json"
validation_config_path = project_root / "config/validation.json"
audit_paths = {
    "phase_01": result_dir / "phase_01_data_quality.json",
    "phase_02": result_dir / "phase_02_feature_quality.json",
    "phase_03": result_dir / "phase_03_supervised_quality.json",
    "phase_04": result_dir / "phase_04_split_audit.json",
}
required_paths = [processed_path, feature_config_path, validation_config_path, *audit_paths.values()]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Thiếu presentation artifact: {missing_paths}"
audits = {}
for phase_name, path in audit_paths.items():
    with path.open(encoding="utf-8") as file:
        audits[phase_name] = json.load(file)
    assert audits[phase_name]["status"] == "PASS"
with feature_config_path.open(encoding="utf-8") as file:
    feature_config = json.load(file)
with validation_config_path.open(encoding="utf-8") as file:
    validation_config = json.load(file)
df_prepared = pd.read_csv(processed_path)
print("Presentation artifacts: Phase 1–4 PASS")

Presentation artifacts: Phase 1–4 PASS


## Phase 1 — Data Preparation

Phase 1 kiểm tra dữ liệu gốc, xử lý trên working copy và tạo timeline theo giờ có thể truy vết. Raw dataset không bị thay đổi.

In [2]:
phase_01 = audits["phase_01"]
phase_01_summary = pd.DataFrame(
    [
        ["Số dòng raw", phase_01["raw_rows"]],
        ["Số dòng prepared", phase_01["prepared_rows"]],
        ["Bắt đầu", phase_01["prepared_time_start"]],
        ["Kết thúc", phase_01["prepared_time_end"]],
        ["Timestamp trùng trước xử lý", phase_01["duplicate_timestamps_before"]],
        ["Timestamp thiếu trước reindex", phase_01["missing_hourly_timestamps_before"]],
        ["Target thiếu sau reindex", phase_01["missing_AEP_MW"]],
        ["Timeline liên tục theo giờ", phase_01["timeline_continuous_hourly"]],
        ["Raw checksum không đổi", phase_01["raw_sha256_before"] == phase_01["raw_sha256_after"]],
        ["Trạng thái", phase_01["status"]],
    ],
    columns=["Chỉ số", "Kết quả"],
)
phase_01_quality = pd.DataFrame(
    [
        ["missing_timestamp", phase_01["source_count_0"]],
        ["observed", phase_01["source_count_1"]],
        ["duplicate_aggregated", phase_01["source_count_2"]],
    ],
    columns=["quality_flag", "Số timestamp"],
)
display(phase_01_summary)
display(phase_01_quality)
display(pd.concat([df_prepared.head(3), df_prepared.tail(3)], ignore_index=True))

,Chỉ số,Kết quả
0,Số dòng raw,121273
1,Số dòng prepared,121296
2,Bắt đầu,2004-10-01 01:00:00
3,Kết thúc,2018-08-03 00:00:00
4,Timestamp trùng trước xử lý,4
5,Timestamp thiếu trước reindex,27
6,Target thiếu sau reindex,27
7,Timeline liên tục theo giờ,True
8,Raw checksum không đổi,True
9,Trạng thái,PASS


,quality_flag,Số timestamp
0,missing_timestamp,27
1,observed,121265
2,duplicate_aggregated,4


,Datetime,AEP_MW,source_count,quality_flag
0,2004-10-01 01:00:00,12379.0,1,observed
1,2004-10-01 02:00:00,11935.0,1,observed
2,2004-10-01 03:00:00,11692.0,1,observed
3,2018-08-02 22:00:00,17001.0,1,observed
4,2018-08-02 23:00:00,15964.0,1,observed
5,2018-08-03 00:00:00,14809.0,1,observed


### Kết luận Phase 1

- Timeline sau xử lý có **121.296 timestamp giờ**, tăng dần và không còn timestamp trùng.
- **4 timestamp trùng** được tổng hợp bằng trung bình; **27 timestamp thiếu** được thêm vào timeline nhưng target vẫn là `NaN`.
- Raw checksum không đổi và timestamp tiếp tục được giữ ở trạng thái timezone-naive.
- **Phase 1: PASS.**

## Phase 2 — Feature Engineering

Phase 2 định nghĩa các lag features và rolling features chỉ từ lịch sử trước thời điểm dự báo.

In [3]:
phase_02 = audits["phase_02"]
phase_02_summary = pd.DataFrame(
    [
        ["Số dòng input/output", phase_02["feature_table"]["rows"]],
        ["Tổng số cột bảng feature", phase_02["feature_table"]["columns"]],
        ["Lag features", phase_02["feature_table"]["lag_feature_count"]],
        ["Rolling features", phase_02["feature_table"]["rolling_feature_count"]],
        ["Lag p", ", ".join(map(str, phase_02["feature_experiments"]["lag_only"]))],
        ["Rolling windows", ", ".join(map(str, feature_config["rolling_features"]["windows"]))],
        ["Shift trước rolling", feature_config["rolling_features"]["shift_before_rolling"]],
        ["Dùng target hiện tại", phase_02["rolling_features"]["current_target_used"]],
        ["Trạng thái", phase_02["status"]],
    ],
    columns=["Chỉ số", "Kết quả"],
)
selected_missing = {
    "lag_1": phase_02["lag_features"]["lag_nan_counts"]["lag_1"],
    "lag_24": phase_02["lag_features"]["lag_nan_counts"]["lag_24"],
    "lag_72": phase_02["lag_features"]["lag_nan_counts"]["lag_72"],
    "lag_168": phase_02["lag_features"]["lag_nan_counts"]["lag_168"],
    **phase_02["rolling_features"]["rolling_nan_counts"],
}
phase_02_missing = pd.DataFrame(
    selected_missing.items(),
    columns=["Feature đại diện", "Số giá trị NaN"],
)
display(phase_02_summary)
display(phase_02_missing)

,Chỉ số,Kết quả
0,Số dòng input/output,121296
1,Tổng số cột bảng feature,176
2,Lag features,168
3,Rolling features,4
4,Lag p,"24, 72, 168"
5,Rolling windows,"24, 168"
6,Shift trước rolling,1
7,Dùng target hiện tại,False
8,Trạng thái,PASS


,Feature đại diện,Số giá trị NaN
0,lag_1,28
1,lag_24,51
2,lag_72,99
3,lag_168,195
4,rolling_mean_24,672
5,rolling_std_24,672
6,rolling_mean_168,4595
7,rolling_std_168,4595


### Kết luận Phase 2

- Đã định nghĩa **168 lag features** và **4 rolling features** cho các cửa sổ 24 và 168 giờ.
- Rolling features sử dụng `shift(1)`, vì vậy không chứa `AEP_MW(t)` hoặc dữ liệu tương lai.
- Phase 2 giữ nguyên missing values để Phase 3 xử lý bằng common validity mask.
- **Phase 2: PASS.**

## Phase 3 — X/y Preparation

Phase 3 tạo sáu supervised configurations trên cùng một timestamp population để việc so sánh model về sau công bằng.

In [4]:
phase_03 = audits["phase_03"]
row_loss = phase_03["row_loss"]
phase_03_row_loss = pd.DataFrame(
    [
        ["Số dòng ban đầu", row_loss["initial_rows"]],
        ["Invalid target", row_loss["reason_counts"]["invalid_target"]],
        ["Invalid lag history", row_loss["reason_counts"]["invalid_lag_history"]],
        ["Invalid rolling history", row_loss["reason_counts"]["invalid_rolling_history"]],
        ["Số dòng loại duy nhất", row_loss["removed_unique_rows"]],
        ["Số dòng hợp lệ", row_loss["usable_rows"]],
        ["Timestamp hợp lệ đầu tiên", row_loss["first_usable_timestamp"]],
        ["Timestamp hợp lệ cuối cùng", row_loss["last_usable_timestamp"]],
    ],
    columns=["Chỉ số", "Kết quả"],
)
configuration_rows = []
for name, values in phase_03["supervised_datasets"]["datasets"].items():
    configuration_rows.append(
        [name, values["samples"], values["features"], values["alignment"], values["finite_X_y"]]
    )
phase_03_configurations = pd.DataFrame(
    configuration_rows,
    columns=["Configuration", "Samples", "Features", "Alignment", "Finite X/y"],
)
display(phase_03_row_loss)
display(phase_03_configurations)

,Chỉ số,Kết quả
0,Số dòng ban đầu,121296
1,Invalid target,27
2,Invalid lag history,4595
3,Invalid rolling history,4595
4,Số dòng loại duy nhất,4621
5,Số dòng hợp lệ,116675
6,Timestamp hợp lệ đầu tiên,2004-10-08 01:00:00
7,Timestamp hợp lệ cuối cùng,2018-08-03 00:00:00


,Configuration,Samples,Features,Alignment,Finite X/y
0,lag_only_p24,116675,24,PASS,True
1,lag_only_p72,116675,72,PASS,True
2,lag_only_p168,116675,168,PASS,True
3,lag_rolling_p24,116675,28,PASS,True
4,lag_rolling_p72,116675,76,PASS,True
5,lag_rolling_p168,116675,172,PASS,True


### Kết luận Phase 3

- Common maximal mask giữ lại **116.675 mẫu** và loại duy nhất **4.621 mẫu** không hợp lệ.
- Cả sáu configurations dùng chung timestamp population; số feature lần lượt là 24, 72, 168, 28, 76 và 172.
- `AEP_MW`, `Datetime`, `source_count` và `quality_flag` không nằm trong model feature set.
- **Phase 3: PASS.**

## Phase 4 — Temporal Split & Validation Design

Phase 4 chia train/test theo thời gian và định nghĩa năm expanding-window CV folds chỉ bên trong top-level train.

In [5]:
phase_04 = audits["phase_04"]
top_level = phase_04["top_level_split"]
phase_04_split = pd.DataFrame(
    [
        ["Train", top_level["train_rows"], top_level["train_start"], top_level["train_end"]],
        ["Final test", top_level["test_rows"], top_level["test_start"], top_level["test_end"]],
    ],
    columns=["Tập dữ liệu", "Samples", "Bắt đầu", "Kết thúc"],
)
fold_rows = []
for fold in phase_04["cross_validation"]["folds"]:
    fold_rows.append(
        [fold["fold"], fold["train_samples"], fold["validation_samples"], fold["train_end"], fold["validation_start"], fold["validation_end"]]
    )
phase_04_folds = pd.DataFrame(
    fold_rows,
    columns=["Fold", "Train samples", "Validation samples", "Train end", "Validation start", "Validation end"],
)
phase_04_policy = pd.DataFrame(
    [
        ["Cutoff", validation_config["test_start"]],
        ["Chiến lược CV", validation_config["cv"]["strategy"]],
        ["Số folds", validation_config["cv"]["n_splits"]],
        ["Validation samples/fold", validation_config["cv"]["test_size"]],
        ["Gap", validation_config["cv"]["gap"]],
        ["Final test sealed", phase_04["final_test"]["sealed"]],
        ["Final-test metrics đã tính", phase_04["final_test"]["metrics_computed"]],
        ["Trạng thái", phase_04["status"]],
    ],
    columns=["Thiết lập", "Giá trị"],
)
display(phase_04_split)
display(phase_04_folds)
display(phase_04_policy)

,Tập dữ liệu,Samples,Bắt đầu,Kết thúc
0,Train,108083,2004-10-08 01:00:00,2017-08-02 23:00:00
1,Final test,8592,2017-08-03 00:00:00,2018-08-03 00:00:00


,Fold,Train samples,Validation samples,Train end,Validation start,Validation end
0,1,97283,2160,2016-05-02 22:00:00,2016-05-02 23:00:00,2016-07-31 22:00:00
1,2,99443,2160,2016-07-31 22:00:00,2016-07-31 23:00:00,2016-10-29 22:00:00
2,3,101603,2160,2016-10-29 22:00:00,2016-10-29 23:00:00,2017-01-27 22:00:00
3,4,103763,2160,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-05-04 23:00:00
4,5,105923,2160,2017-05-04 23:00:00,2017-05-05 00:00:00,2017-08-02 23:00:00


,Thiết lập,Giá trị
0,Cutoff,2017-08-03 00:00:00
1,Chiến lược CV,expanding_window
2,Số folds,5
3,Validation samples/fold,2160
4,Gap,0
5,Final test sealed,True
6,Final-test metrics đã tính,False
7,Trạng thái,PASS


### Kết luận Phase 4

- Top-level train có **108.083 mẫu**; sealed final test có **8.592 mẫu**.
- Năm CV folds đều nằm hoàn toàn trong train, mỗi fold có **2.160 validation samples** và training window mở rộng dần.
- Sáu feature configurations sử dụng cùng train/test indices và cùng CV folds.
- Final test chưa được dùng để chọn feature, tuning, chọn model hoặc tính metric.
- **Phase 4: PASS.**

## Tổng kết Phase 1–4

In [6]:
phase_summary = pd.DataFrame(
    [
        ["Phase 1", "Data Preparation", audits["phase_01"]["status"], "Timeline giờ đã chuẩn bị và có quality metadata"],
        ["Phase 2", "Feature Engineering", audits["phase_02"]["status"], "Lag và rolling features dùng lịch sử quá khứ"],
        ["Phase 3", "X/y Preparation", audits["phase_03"]["status"], "Sáu configurations dùng chung 116.675 mẫu"],
        ["Phase 4", "Temporal Split & Validation", audits["phase_04"]["status"], "Train/CV đã chốt; final test vẫn sealed"],
    ],
    columns=["Phase", "Phạm vi", "Trạng thái", "Kết quả chính"],
)
display(phase_summary)

,Phase,Phạm vi,Trạng thái,Kết quả chính
0,Phase 1,Data Preparation,PASS,Timeline giờ đã chuẩn bị và có quality metadata
1,Phase 2,Feature Engineering,PASS,Lag và rolling features dùng lịch sử quá khứ
2,Phase 3,X/y Preparation,PASS,Sáu configurations dùng chung 116.675 mẫu
3,Phase 4,Temporal Split & Validation,PASS,Train/CV đã chốt; final test vẫn sealed


### Trạng thái tiếp theo

Phase 1–4 đã hoàn thành và cung cấp input đã kiểm chứng cho Phase 5. Notebook này chưa trình bày model, prediction hoặc metric vì Phase 5–7 chưa được triển khai. Final test tiếp tục ở trạng thái **SEALED** cho đến Phase 6.